# Lab 14 - Detecção de Objetos em Imagens e Vídeos

Neste laboratório não treinaremos nenhum modelo. Usaremos um detector YOLO pré-treinado para ler bounding boxes, classes e confianças em uma imagem e, depois, criar uma aplicação simples de contagem em vídeo.

## Objetivos

1. Interpretar a saída de um detector: caixa, classe e confiança.
2. Ajustar o limiar de confiança e observar seu efeito.
3. Processar um vídeo quadro a quadro.
4. Usar IDs de rastreamento para contar objetos que cruzam uma linha virtual.

## Antes de começar

No Google Colab, selecione **Runtime > Change runtime type > T4 GPU**. O laboratório também funciona em CPU, mas o processamento de vídeo será mais lento.

A palavra `pretrained` significa que os pesos já foram treinados com imagens do conjunto COCO. Nesta aula, vamos **usar e interpretar** o modelo, não alterar seus pesos.

In [ ]:
%pip install -q ultralytics opencv-python-headless matplotlib

## Parte 1 - Carregando um detector pré-treinado

Usaremos `YOLO`, da biblioteca Ultralytics. A escolha do modelo `yolo26n.pt` segue um critério prático:

- `YOLO`: família de detectores de objetos;
- `26`: versão usada nesta disciplina;
- `n`: *nano*, a menor variante, escolhida para rodar bem no Colab;
- `.pt`: arquivo de pesos PyTorch.

A arquitetura não é o foco agora. Trate o modelo como um detector que recebe uma imagem e devolve uma lista de hipóteses.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
MODEL_NAME = "yolo26n.pt"

print(f"Dispositivo: {DEVICE}")
model = YOLO(MODEL_NAME)
print(f"Modelo carregado: {MODEL_NAME}")
print(f"Exemplos de classes COCO: {list(model.names.items())[:8]}")

## Parte 2 - Detectando objetos em uma imagem

A célula abaixo baixa uma imagem de teste. Você pode substituir `IMAGE_PATH` pelo caminho de uma imagem sua. Primeiro faremos a inferência; depois veremos os valores retornados pelo modelo antes de desenhar qualquer caixa.

In [ ]:
from urllib.request import urlretrieve

IMAGE_PATH = Path("street.jpg")
if not IMAGE_PATH.exists():
    urlretrieve("https://ultralytics.com/images/bus.jpg", IMAGE_PATH)

confidence_threshold = 0.40
result = model(str(IMAGE_PATH), conf=confidence_threshold, device=DEVICE, verbose=False)[0]

image_rgb = cv2.cvtColor(cv2.imread(str(IMAGE_PATH)), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 7))
plt.imshow(image_rgb)
plt.axis("off")
plt.title("Imagem de entrada")
plt.show()

In [ ]:
boxes_xyxy = result.boxes.xyxy.cpu().numpy()
confidences = result.boxes.conf.cpu().numpy()
class_ids = result.boxes.cls.cpu().numpy().astype(int)

print(f"Detecções com confiança >= {confidence_threshold:.0%}: {len(boxes_xyxy)}\n")
for box, confidence, class_id in zip(boxes_xyxy, confidences, class_ids):
    x1, y1, x2, y2 = box.round().astype(int)
    print(
        f"classe={model.names[class_id]:10s} | confiança={confidence:.1%} | "
        f"caixa=[{x1}, {y1}, {x2}, {y2}]"
    )

Cada linha é uma detecção. A caixa está no formato `[x1, y1, x2, y2]`: canto superior esquerdo e canto inferior direito. Agora podemos pedir ao YOLO para desenhar as mesmas informações.

**Experimento:** execute as células anteriores usando `confidence_threshold = 0.20`, `0.40` e `0.70`. Quais objetos aparecem ou desaparecem?

In [ ]:
annotated_rgb = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 7))
plt.imshow(annotated_rgb)
plt.axis("off")
plt.title(f"YOLO com confiança mínima de {confidence_threshold:.0%}")
plt.show()

## Parte 3 - Preparando um vídeo

Envie um vídeo curto para o Colab. Bons exemplos são pessoas em um corredor, carros em uma rua ou objetos em uma esteira. Escolha uma classe COCO que apareça no vídeo.

Uma detecção em vídeo ainda é feita quadro a quadro. Para saber se a pessoa no quadro atual é a mesma pessoa do quadro anterior, usaremos rastreamento com IDs persistentes.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    VIDEO_PATH = Path(next(iter(uploaded)))
except ImportError:
    VIDEO_PATH = Path("video.mp4")

if not VIDEO_PATH.exists():
    raise FileNotFoundError("Envie um video no Colab ou defina VIDEO_PATH para um arquivo local.")

print(f"Vídeo selecionado: {VIDEO_PATH}")

## Parte 4 - Observando o rastreamento

Antes de contar, vamos observar o que o rastreador acrescenta à detecção. `model.track(..., persist=True)` tenta atribuir o mesmo ID ao mesmo objeto em quadros consecutivos.

Execute a célula seguinte e acompanhe os rótulos. Uma pessoa que permanece no vídeo deve manter, em geral, o mesmo número. O ID não é a classe: várias pessoas podem ser `person`, mas cada uma deve ter seu próprio ID.

In [ ]:
TARGET_CLASS = "person"
CONFIDENCE_THRESHOLD = 0.40
TRACKING_SECONDS = 5

target_class_id = next(
    class_id for class_id, name in model.names.items() if name == TARGET_CLASS
)
capture = cv2.VideoCapture(str(VIDEO_PATH))
fps = capture.get(cv2.CAP_PROP_FPS) or 30
max_frames = int(fps * TRACKING_SECONDS)

frames_with_ids = []
for _ in range(max_frames):
    success, frame = capture.read()
    if not success:
        break

    tracked = model.track(
        frame,
        persist=True,
        classes=[target_class_id],
        conf=CONFIDENCE_THRESHOLD,
        device=DEVICE,
        verbose=False,
    )[0]
    frames_with_ids.append(cv2.cvtColor(tracked.plot(), cv2.COLOR_BGR2RGB))

capture.release()
if not frames_with_ids:
    raise RuntimeError("O video nao possui quadros que possam ser processados.")

print(f"Quadros processados: {len(frames_with_ids)}")
print("Observe os rotulos: o mesmo objeto deve manter o mesmo ID entre quadros.")

sample_indices = np.linspace(0, len(frames_with_ids) - 1, 4, dtype=int)
fig, axes = plt.subplots(1, len(sample_indices), figsize=(16, 4))
for axis, frame_index in zip(axes, sample_indices):
    axis.imshow(frames_with_ids[frame_index])
    axis.set_title(f"Quadro {frame_index}")
    axis.axis("off")
plt.tight_layout()
plt.show()

## Parte 5 - Contando objetos que cruzam uma linha

Agora que sabemos para que serve um ID persistente, podemos contar. Escolha a classe que será contada. `person` é um bom primeiro teste. A linha horizontal fica no meio do quadro. Um objeto é contado apenas uma vez quando seu centro cruza a linha de cima para baixo.

A chamada `model.track(..., persist=True)` continua sendo usada porque `box.id` nos permite guardar a posição anterior de cada objeto e impedir contagens repetidas.

In [ ]:
TARGET_CLASS = "person"
CONFIDENCE_THRESHOLD = 0.40

target_class_id = next(class_id for class_id, name in model.names.items() if name == TARGET_CLASS)
capture = cv2.VideoCapture(str(VIDEO_PATH))
fps = capture.get(cv2.CAP_PROP_FPS) or 30
width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
line_y = height // 2

OUTPUT_PATH = Path("video_contagem.mp4")
writer = cv2.VideoWriter(
    str(OUTPUT_PATH),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height),
)

previous_center_y = {}
counted_ids = set()
count = 0

while capture.isOpened():
    success, frame = capture.read()
    if not success:
        break

    tracked = model.track(
        frame,
        persist=True,
        classes=[target_class_id],
        conf=CONFIDENCE_THRESHOLD,
        device=DEVICE,
        verbose=False,
    )[0]
    annotated = tracked.plot()

    if tracked.boxes.id is not None:
        boxes = tracked.boxes.xyxy.cpu().numpy()
        track_ids = tracked.boxes.id.int().cpu().tolist()
        for box, track_id in zip(boxes, track_ids):
            _, y1, _, y2 = box
            center_y = (y1 + y2) / 2
            previous_y = previous_center_y.get(track_id)

            crossed_down = previous_y is not None and previous_y < line_y <= center_y
            if crossed_down and track_id not in counted_ids:
                count += 1
                counted_ids.add(track_id)

            previous_center_y[track_id] = center_y

    cv2.line(annotated, (0, line_y), (width, line_y), (0, 255, 255), 3)
    cv2.putText(
        annotated,
        f"{TARGET_CLASS}: {count}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        3,
    )
    writer.write(annotated)

capture.release()
writer.release()
print(f"Contagem final: {count}")
print(f"Vídeo salvo em: {OUTPUT_PATH}")

In [ ]:
from IPython.display import Video

Video(str(OUTPUT_PATH), embed=True)

## Desafio

1. Troque `TARGET_CLASS` por outra classe que exista no vídeo.
2. Mova a linha de contagem para `height // 3` ou `2 * height // 3`. A contagem muda?
3. Use os limiares `0.20`, `0.40` e `0.70`. Qual deles produz a melhor contagem para seu vídeo?
4. Modifique a regra para contar objetos que cruzam de baixo para cima.
5. Explique um erro observado. Foi uma falha de detecção, de rastreamento ou da posição da linha?

